# ML Assignment 1
Muneera Aharbi - 202002460

# Part 1

Q1. What is feature scaling, and why is it important? Name two common techniques.

Feature scaling is the process of transforming numeric features so that they are on comparable ranges. It is important because many machine learning algorithms are sensitive to feature magnitude, especially algorithms that use distances or gradient descent.

Two common techniques are standardization and normalization. Standardization transforms a feature to have mean 0 and standard deviation 1. Normalization, also called min-max scaling, transforms values into a fixed range such as 0 to 1.


Q2. How can a transformer be added in a preparation pipeline to select only the most important attributes?
A feature selection transformer can be inserted inside the scikit-learn pipeline after preprocessing. For example, SelectKBest can select the top k features using f_classif for classification or f_regression for regression. This makes feature selection part of the pipeline, so it is applied consistently during training and testing.



In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("select", SelectKBest(score_func=f_classif, k=10)),
    ("model", LogisticRegression(max_iter=1000))
])

Q3. Key differences between data preparation and data preprocessing.

Data preparation is the broader stage of getting raw data ready for modeling. It includes collecting data, understanding it, splitting it into training and test sets, handling missing values, and deciding which columns are useful.

Data preprocessing is a more specific technical step inside preparation. It includes transformations such as imputation, encoding categorical features, scaling numerical features, and feature selection.
Example of preparation: splitting data into training and test sets.

Example of preprocessing: applying OneHotEncoder to categorical
variables.


Q4. Holdout validation vs cross-validation.

Holdout validation splits the data once into a training set and validation/test set. It is simple and fast, so it is useful when the dataset is large.

Cross-validation splits the data into several folds and trains/evaluates the model multiple times. It gives a more reliable estimate of performance, especially when the dataset is small or when model comparison is important.


 Q5. Purpose of feature engineering with examples.

Feature engineering creates new useful features from existing data to help the model find stronger patterns.

Example 1: In Titanic, FamilySize = SibSp + Parch + 1 can improve survival prediction because family structure may affect survival.

Example 2: In House Prices, AgeOfHouse = YrSold - YearBuilt can improve price prediction because newer houses may have different values than older ones.


Q6. Stratified sampling and why it matters in classification.

Stratified sampling preserves the same class proportions in the training and test sets as in the original dataset. It is especially important in classification when the classes are imbalanced.

For example, if only a small percentage of passengers survived, stratified sampling ensures both the training and test sets contain a similar survival ratio, which gives a fairer evaluation.


# Part 2

Shared imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    mean_squared_error, r2_score, classification_report
)


Titanic Dataset: Classification

In [ ]:
# Task 1: Load and explore

titanic = pd.read_csv("titanic/train.csv")   # replace path if needed
print(titanic.head())
print(titanic.info())
print(titanic.describe(include="all"))
print(titanic.isna().sum())

# Missing values strategy:
# Age: median imputation because it is numeric and may contain outliers.
# Embarked: most frequent imputation because it is categorical.
# Cabin: many values are missing, so we drop it for the basic model.
# Name/Ticket: not directly useful in raw form, but Name will be used later for Title feature.
# Task 2: Split features and target using stratified sampling

titanic_basic = titanic.drop(columns=["Cabin", "Ticket", "Name"], errors="ignore")
X = titanic_basic.drop("Survived", axis=1)
y = titanic_basic["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(y.value_counts(normalize=True))
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

# Stratified sampling is appropriate because Survived is a classification target
# and we want both sets to preserve the survival/non-survival ratio.
# Task 3: Processing pipeline + logistic regression

num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

titanic_model = Pipeline([
    ("preprocess", preprocessor),
    ("select", SelectKBest(score_func=f_classif, k="all")),
    ("model", LogisticRegression(max_iter=1000))
])

titanic_model.fit(X_train, y_train)
y_pred = titanic_model.predict(X_test)
print(classification_report(y_test, y_pred))
# Task 4: Evaluation and baseline comparison

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print("Logistic Regression:", {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1})

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)
print("Baseline accuracy:", accuracy_score(y_test, base_pred))
print("Baseline F1:", f1_score(y_test, base_pred, zero_division=0))
# Task 5: Feature engineering: FamilySize and Title

def add_titanic_features(df):
    df = df.copy()
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    if "Name" in df.columns:
        df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
        df["Title"] = df["Title"].replace({
            "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
            "Lady": "Rare", "Countess": "Rare", "Capt": "Rare",
            "Col": "Rare", "Don": "Rare", "Dr": "Rare",
            "Major": "Rare", "Rev": "Rare", "Sir": "Rare", "Jonkheer": "Rare"
        })
    return df

    titanic_fe = add_titanic_features(titanic).drop(columns=["Cabin", "Ticket", "Name"], errors="ignore")
X_fe = titanic_fe.drop("Survived", axis=1)
y_fe = titanic_fe["Survived"]

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe, y_fe, test_size=0.2, random_state=42, stratify=y_fe
)

num_features_fe = X_train_fe.select_dtypes(include=[np.number]).columns.tolist()
cat_features_fe = X_train_fe.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor_fe = ColumnTransformer([
    ("num", num_pipeline, num_features_fe),
    ("cat", cat_pipeline, cat_features_fe)
])

titanic_model_fe = Pipeline([
    ("preprocess", preprocessor_fe),
    ("select", SelectKBest(score_func=f_classif, k="all")),
    ("model", LogisticRegression(max_iter=1000))
])

titanic_model_fe.fit(X_train_fe, y_train_fe)
y_pred_fe = titanic_model_fe.predict(X_test_fe)
print(classification_report(y_test_fe, y_pred_fe))
print("Feature-engineered F1:", f1_score(y_test_fe, y_pred_fe))



House Prices Dataset: Regression

In [ ]:
# Task 1: Load and explore

house = pd.read_csv("house/train.csv")   # replace path if needed
print(house.head())
print(house.info())
print(house.describe(include="all"))
print(house.isna().sum().sort_values(ascending=False).head(20))

# Missing values strategy:
# Numeric columns: median imputation.
# Categorical columns: most frequent imputation.
# Columns with many missing values may also be dropped if they add little value.
# Task 2: Split features and target using random sampling

X = house.drop("SalePrice", axis=1)
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Random sampling is acceptable because this is a regression target, not a class label.
# Task 3: Processing pipeline + Linear Regression

num_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_features),
    ("cat", cat_pipeline, cat_features)
])

house_model = Pipeline([
    ("preprocess", preprocessor),
    ("select", SelectKBest(score_func=f_regression, k=50)),
    ("model", LinearRegression())
])

house_model.fit(X_train, y_train)
y_pred = house_model.predict(X_test)
# Task 4: Evaluation and baseline comparison

rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
print("Linear Regression RMSE:", rmse)
print("Linear Regression R2:", r2)

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)
print("Baseline RMSE:", mean_squared_error(y_test, base_pred, squared=False))
print("Baseline R2:", r2_score(y_test, base_pred))
# Task 5: Feature engineering: AgeOfHouse and TotalBathrooms

def add_house_features(df):
    df = df.copy()
    if "YrSold" in df.columns and "YearBuilt" in df.columns:
        df["AgeOfHouse"] = df["YrSold"] - df["YearBuilt"]
    bath_cols = ["FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"]
    for col in bath_cols:
        if col not in df.columns:
            df[col] = 0
    df["TotalBathrooms"] = df["FullBath"] + 0.5*df["HalfBath"] + df["BsmtFullBath"] + 0.5*df["BsmtHalfBath"]
    return df

house_fe = add_house_features(house)
X_fe = house_fe.drop("SalePrice", axis=1)
y_fe = house_fe["SalePrice"]

X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(
    X_fe, y_fe, test_size=0.2, random_state=42
)

num_features_fe = X_train_fe.select_dtypes(include=[np.number]).columns.tolist()
cat_features_fe = X_train_fe.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor_fe = ColumnTransformer([
    ("num", num_pipeline, num_features_fe),
    ("cat", cat_pipeline, cat_features_fe)
])

house_model_fe = Pipeline([
    ("preprocess", preprocessor_fe),
    ("select", SelectKBest(score_func=f_regression, k=50)),
    ("model", LinearRegression())
])

house_model_fe.fit(X_train_fe, y_train_fe)
y_pred_fe = house_model_fe.predict(X_test_fe)
print("Feature-engineered RMSE:", mean_squared_error(y_test_fe, y_pred_fe, squared=False))
print("Feature-engineered R2:", r2_score(y_test_fe, y_pred_fe))
